# 11 — Lectores de mediciones SQLite

El DataServer de IviumSoft escribe cada medición en un archivo SQLite
(`DataServer_*.idf.sqlite`) y mantiene un catálogo (`index.sqlite`) de ellas, por defecto
en `C:\\IviumStat\\DataServer\\measurements`. La capa `Tools` los lee en **solo lectura**
y de forma **segura con WAL**, así que incluso puedes seguir una medición mientras IviumSoft
aún la está escribiendo — y leer no necesita driver ni hardware.

### Qué puedes hacer

- Explorar el catálogo y filtrar por número de serie, técnica o fecha
- Abrir una medición y leer sus metadatos, método, partes y puntos
- Decodificar el byte de estado por punto (sobrecargas, rango de corriente)
- Leer puntos de impedancia (EIS)
- Seguir una medición en curso de forma incremental (`after_point_id` / `latest_point_id`)
- Exportar a CSV (y a un DataFrame de pandas si pandas está instalado)

### Apúntalo a tus datos

Este notebook lee los **archivos de medición reales de tu máquina**. Ajusta `DATA_SERVER_DIR`
abajo a tu carpeta de mediciones del DataServer. Cada celda de datos se protege sola si la
carpeta no existe, así que puedes leer el notebook aunque no tengas IviumSoft instalado.

In [ ]:
from pathlib import Path

from pyvium.tools import MeasurementReader, MeasurementIndex
print("pyvium.tools SQLite readers imported")

## 1. Apunta a tu DataServer

`index.sqlite` vive en la carpeta de mediciones; el archivo `DataServer_*.idf.sqlite` de cada
medición se localiza por su fila del catálogo (`path` + `file`), resuelto bajo esta misma
carpeta. Cambia `DATA_SERVER_DIR` si tu instalación difiere.

In [ ]:
# Ubicación por defecto de IviumSoft; edítala para tu máquina.
DATA_SERVER_DIR = Path(r"C:\IviumStat\DataServer\measurements")
INDEX_PATH = DATA_SERVER_DIR / "index.sqlite"

HAVE_INDEX = INDEX_PATH.exists()
if HAVE_INDEX:
    print("using catalog:", INDEX_PATH)
else:
    print(f"No index.sqlite at {INDEX_PATH}")
    print("Edit DATA_SERVER_DIR above to point at your DataServer measurements folder.")

## 2. Explorar el catálogo

`MeasurementIndex` lee `index.sqlite` y devuelve entradas de más reciente a más antigua.
Filtra por serie (sin distinguir mayúsculas), técnica, subcadena de título, proyecto/operador
y rango de fechas.

In [ ]:
entry = None
if HAVE_INDEX:
    with MeasurementIndex(str(INDEX_PATH)) as index:
        recent = index.entries(limit=10)
        for e in recent:
            print(f"  {e.start_time}  {str(e.technique):<20} serial={e.serialnumber}  {e.file}")

        # Filtros de ejemplo (descomenta y adapta):
        # index.entries(serialnumber="P33162")
        # index.entries(technique="CyclicVoltammetry")
        # index.entries(start_after="2024-01-01", start_before="2024-12-31")

    entry = recent[0] if recent else None
    print("\nselected:", entry.file if entry else "catalog is empty")
else:
    print("skipped - no catalog (set DATA_SERVER_DIR in cell 1)")

## 3. Abrir la medición seleccionada

`resolve_path(entry, base_dir)` construye la ruta del archivo desde la fila del catálogo;
`open_measurement(entry, base_dir)` devuelve un `MeasurementReader` (aún sin abrir).
`base_dir` es la carpeta de mediciones.

In [ ]:
MEASUREMENT_PATH = None
if entry is not None:
    resolved = MeasurementIndex.resolve_path(entry, str(DATA_SERVER_DIR))
    print("measurement file:", resolved)
    if Path(resolved).exists():
        MEASUREMENT_PATH = resolved
        with MeasurementReader(MEASUREMENT_PATH) as reader:
            print("database version :", reader.database_version)
            print("metadata         :", reader.metadata())
            print("measurements     :", reader.measurements())
            print("method params    :", reader.method_parameters())
            for part in reader.measurement_parts()[:5]:
                print("  part:", part)
    else:
        print("File not found - adjust base_dir (resolve_path) to match your layout.")
else:
    print("no measurement selected")

## 4. Leer puntos (con estado decodificado)

`read_points()` une cada punto con el contexto de su parte (ciclo / nivel / canal) y expone
un `status` decodificado (banderas de sobrecarga e índice de rango de corriente) a partir del
`statusbyte`.

In [ ]:
if MEASUREMENT_PATH:
    with MeasurementReader(MEASUREMENT_PATH) as reader:
        points = reader.read_points()
    print(f"{len(points)} points; first 5:")
    for p in points[:5]:
        print(f"  id={p.point_id} t={p.t} x={p.x} y={p.y} z={p.z} "
              f"cycle={p.cycle} level={p.level} status={p.status}")
else:
    print("no measurement selected")

## 5. Seguir una medición en curso

Mientras IviumSoft aún escribe, consulta `latest_point_id()` y pasa el último id que
consumiste a `read_points(after_point_id=...)` para obtener solo los puntos nuevos. El lector
abre en solo lectura y de forma segura con WAL, así que nunca bloquea al escritor.

In [ ]:
if MEASUREMENT_PATH:
    with MeasurementReader(MEASUREMENT_PATH) as reader:
        latest = reader.latest_point_id()
        print("latest point id:", latest)
        if latest and latest > 5:
            newer = reader.read_points(after_point_id=latest - 5)
            print(f"points after id {latest - 5}:", [p.point_id for p in newer])
else:
    print("no measurement selected")

## 6. Puntos de impedancia (EIS)

Para técnicas EIS, la tabla `pointfra` contiene la respuesta en frecuencia; `read_impedance()`
devuelve `ImpedancePoint`s. Para mediciones no-EIS (sin `pointfra`) devuelve una lista vacía,
así que no necesitas conocer la técnica de antemano.

In [ ]:
if MEASUREMENT_PATH:
    with MeasurementReader(MEASUREMENT_PATH) as reader:
        eis = reader.read_impedance()
    if eis:
        for z in eis[:5]:
            print(f"  id={z.point_id}  f={z.frequency} Hz  Z'={z.z_re}  Z''={z.z_im}  "
                  f"quality={z.quality}")
    else:
        print("no impedance points (not an EIS measurement)")
else:
    print("no measurement selected")

## 7. Exportar

`to_csv()` escribe todos los puntos; `to_dataframe()` devuelve un DataFrame de pandas (pandas
se importa de forma perezosa, así que solo se necesita si lo llamas).

In [ ]:
if MEASUREMENT_PATH:
    import tempfile

    out_csv = Path(tempfile.gettempdir()) / "measurement_points.csv"
    with MeasurementReader(MEASUREMENT_PATH) as reader:
        reader.to_csv(str(out_csv))
    print("wrote", out_csv)
    for line in out_csv.read_text(encoding="utf-8").splitlines()[:3]:
        print("  ", line)

    try:
        with MeasurementReader(MEASUREMENT_PATH) as reader:
            dataframe = reader.to_dataframe()
        print(dataframe.head())
    except ImportError:
        print("pandas not installed - skipping to_dataframe()")
else:
    print("no measurement selected")

## 8. Atajo: la última medición, directamente desde IviumSoft

En lugar de explorar el catálogo, `Pyvium.get_db_file_name()` devuelve la ruta completa de la
base de datos de medición **creada más recientemente**. Esta necesita el driver abierto e
IviumSoft en ejecución (es la única celda aquí que habla con el driver); los lectores en sí no.

In [ ]:
from pyvium import Pyvium

try:
    Pyvium.open_driver()
    db_path = Pyvium.get_db_file_name()
    print("last created DB:", db_path)
    if db_path and Path(db_path).exists():
        with MeasurementReader(db_path) as reader:
            print("technique:", reader.method_parameters().get("Technique"))
            print("points:", len(reader.read_points()))
except Exception as error:
    print(f"skipped ({type(error).__name__}: {error}) - needs IviumSoft running")
finally:
    try:
        Pyvium.close_driver()
    except Exception:
        pass

---

## Resumen

| Tarea | API |
|------|-----|
| Abrir el catálogo | `MeasurementIndex(index_path)` (carpeta por defecto `C:\\IviumStat\\DataServer\\measurements`) |
| Explorar / filtrar | `.entries(serialnumber=..., technique=..., start_after=..., limit=...)` |
| Entrada -> ruta de archivo | `MeasurementIndex.resolve_path(entry, base_dir)` |
| Entrada -> lector | `.open_measurement(entry, base_dir)` |
| Abrir un archivo de medición | `MeasurementReader(path)` (gestor de contexto) |
| Metadatos / método / partes | `.metadata()`, `.method_parameters()`, `.measurement_parts()` |
| Puntos (estado decodificado) | `.read_points()`, `point.status` |
| Seguir una ejecución en vivo | `.latest_point_id()`, `.read_points(after_point_id=...)` |
| Impedancia | `.read_impedance()` |
| Exportar | `.to_csv(path)`, `.to_dataframe()` |
| Última BD creada (necesita IviumSoft) | `Pyvium.get_db_file_name()` |

## Siguiente

- **`10_instance_lifecycle_management.ipynb`** — lanzar, adoptar y cerrar instancias de IviumSoft
- **`07_data_processing.ipynb`** — analizar archivos IDF y exportar a CSV